In [2]:
# %% [markdown]
# # 3 循环神经网络
# ## 3.1 理论计算题：线性RNN BPTT求∂L/∂W_hh，梯度消失爆炸条件
# ### 模型定义（无偏置）
# h_t = W_hh h_{t-1} + W_hx x_t
# o_t = W_oh h_t
# 损失 L = 1/2 * Σ_{t=1}^T ||o_t - y_t||²
# 令 δ_t = ∂L/∂h_t = W_oh^T (o_t - y_t)
# BPTT递推：∂L/∂h_{t-1} = W_hh^T δ_t
# 展开到t=1：∂L/∂h_{s} = (W_hh^T)^{T-s} δ_T
# 链式求导 W_hh：
# ∂L/∂W_hh = Σ_{t=2}^T δ_t h_{t-1}^T
# 其中 δ_t = W_oh^T(o_t-y_t) + W_hh^T δ_{t+1}, δ_{T+1}=0
#
# ### 梯度消失/爆炸条件
# 梯度传播依赖矩阵幂 (W_hh^T)^k 的谱半径 ρ(W_hh)
# 1. 若 ρ(W_hh) < 1：时间步拉长后梯度指数衰减 → 梯度消失
# 2. 若 ρ(W_hh) > 1：时间步拉长后梯度指数放大 → 梯度爆炸

# %% [markdown]
# ## 3.2 编程题：RNN单步前向+反向传播（tanh激活）
# %%
import numpy as np

def tanh(x):
    return np.tanh(x)

def tanh_grad(x):
    return 1 - np.tanh(x)**2

# 前向传播
def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    # x_t: (B, D_in), h_prev: (B, H)
    # W_hx: (H, D_in), W_hh: (H, H), b_h: (H,)
    pre_h = W_hx @ x_t.T + W_hh @ h_prev.T + b_h[:, None]
    pre_h = pre_h.T  # (B, H)
    h_t = tanh(pre_h)
    cache = (x_t, h_prev, W_hx, W_hh, b_h, pre_h)
    return h_t, cache

# 反向传播
def rnn_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, b_h, pre_h = cache
    B, H = dh_next.shape
    # dpre_h = dh_next * tanh'(pre_h)
    dpre_h = dh_next * tanh_grad(pre_h)  # (B, H)
    
    # db_h
    db_h = np.sum(dpre_h, axis=0)  # (H,)
    # dW_hh
    dW_hh = dpre_h.T @ h_prev     # (H, H)
    # dW_hx
    dW_hx = dpre_h.T @ x_t        # (H, D_in)
    # dh_prev
    dh_prev = dpre_h @ W_hh       # (B, H)
    # dx_t
    dx_t = dpre_h @ W_hx          # (B, D_in)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试
if __name__ == "__main__":
    B, D_in, H = 2, 3, 4
    x_t = np.random.randn(B, D_in)
    h_prev = np.random.randn(B, H)
    W_hx = np.random.randn(H, D_in)
    W_hh = np.random.randn(H, H)
    b_h = np.random.randn(H)
    h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    dh_next = np.random.randn(B, H)
    dx, dh_prev, dWhx, dWhh, db = rnn_backward(dh_next, cache)
    print("h_t shape:", h_t.shape)
    print("dx_t shape:", dx.shape)
    print("dW_hh shape:", dWhh.shape)

h_t shape: (2, 4)
dx_t shape: (2, 3)
dW_hh shape: (4, 4)
